# 04 - Baseline : TF-IDF + Régression logistique

**Projet :** Détection de prompt injection et jailbreak — **Membre 1**

## Objectif
Construire la **baseline** exigée par le cahier des charges : un modèle simple qui sert de point de comparaison pour le modèle deep learning (Membre 2).

## Rappel des notions
- **TF-IDF** : transforme chaque texte en un vecteur de nombres. Un mot fréquent dans ce texte mais rare dans l'ensemble des textes reçoit un poids élevé.
- **Régression logistique** : calcule, à partir de ces nombres, la probabilité que le texte soit malveillant. Si p ≥ 0,5 → malveillant.
- **Taux de détection (recall)** = attaques détectées / attaques totales.
- **Taux de faux positifs (FPR)** = prompts légitimes bloqués à tort / prompts légitimes totaux.

## Règle d'or
Le modèle est entraîné **uniquement sur `train_seen.csv`**. On ne touche **jamais** aux tests pour régler quoi que ce soit : les réglages se font sur `val_seen.csv`.

In [ ]:
import os, joblib
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay)

while not os.path.exists("data/processed") and os.getcwd() != "/":
    os.chdir("..")
print("Racine du dépôt :", os.getcwd())
for d in ("models", "results", "results/figures"):
    os.makedirs(d, exist_ok=True)

SEED = 42

## 1. Chargement des données (créées par le notebook 03)

In [ ]:
P = "data/processed"
train_seen  = pd.read_csv(f"{P}/train_seen.csv")
val_seen    = pd.read_csv(f"{P}/val_seen.csv")
test_seen   = pd.read_csv(f"{P}/test_seen.csv")
test_unseen = pd.read_csv(f"{P}/test_unseen_family.csv")

for name, d in [("train_seen", train_seen), ("val_seen", val_seen), ("test_seen", test_seen), ("test_unseen", test_unseen)]:
    print(f"{name:12s} {len(d):6d} lignes | % malveillants = {100*d.label_binary.mean():.1f}")

## 2. Fonction d'évaluation

On calcule les métriques demandées : **taux de détection**, **taux de faux positifs**, plus précision, F1 et accuracy.
`confusion_matrix` renvoie `[[TN, FP], [FN, TP]]` :
- TN = légitime bien classé, FP = légitime bloqué à tort (faux positif),
- FN = attaque manquée, TP = attaque détectée.

In [ ]:
def evaluate(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "n": len(y_true),
        "accuracy":       accuracy_score(y_true, y_pred),
        "precision":      precision_score(y_true, y_pred, zero_division=0),
        "detection_rate": tp / (tp + fn) if (tp + fn) else float("nan"),   # recall
        "fpr":            fp / (fp + tn) if (fp + tn) else float("nan"),   # faux positifs
        "f1":             f1_score(y_true, y_pred, zero_division=0),
        "TP": int(tp), "FN": int(fn), "FP": int(fp), "TN": int(tn),
    }

def fit_baseline(train_df, ngram_range=(1, 2), analyzer="word", C=1.0):
    vec = TfidfVectorizer(lowercase=True, ngram_range=ngram_range, analyzer=analyzer,
                          min_df=2, max_features=100_000, sublinear_tf=True)
    X = vec.fit_transform(train_df["text"].astype(str))
    clf = LogisticRegression(C=C, max_iter=2000, class_weight="balanced", random_state=SEED)
    clf.fit(X, train_df["label_binary"])
    return vec, clf

def predict(vec, clf, df, threshold=0.5):
    proba = clf.predict_proba(vec.transform(df["text"].astype(str)))[:, 1]
    return (proba >= threshold).astype(int), proba

## 3. Choix des hyperparamètres sur la **validation** uniquement

On compare trois façons de découper le texte en « morceaux » : mots seuls, mots + paires de mots, et groupes de caractères.
Le meilleur F1 sur `val_seen` est retenu. **Les tests ne sont pas utilisés ici.** Cette comparaison est aussi notre première ablation.

In [ ]:
configs = {
    "mots (1)":            dict(ngram_range=(1, 1), analyzer="word"),
    "mots (1-2)":          dict(ngram_range=(1, 2), analyzer="word"),
    "caractères (3-5)":    dict(ngram_range=(3, 5), analyzer="char_wb"),
}
rows = []
for name, cfg in configs.items():
    vec, clf = fit_baseline(train_seen, **cfg)
    pred, _ = predict(vec, clf, val_seen)
    m = evaluate(val_seen.label_binary, pred)
    rows.append({"config": name, **{k: m[k] for k in ["f1", "detection_rate", "fpr"]}})
    print(name, "F1 val =", round(m["f1"], 4))
ablation_ngram = pd.DataFrame(rows)
display(ablation_ngram)
ablation_ngram.to_csv("results/baseline_ablation_ngrams.csv", index=False)

best_name = ablation_ngram.sort_values("f1", ascending=False).iloc[0]["config"]
print("\nConfiguration retenue :", best_name)

## 4. Entraînement final de la baseline (train_seen) et évaluation sur les 3 ensembles

In [ ]:
vec, clf = fit_baseline(train_seen, **configs[best_name])

results = {}
preds = {}
for name, df in [("validation (seen)", val_seen), ("test_seen", test_seen), ("test_unseen_family", test_unseen)]:
    pred, proba = predict(vec, clf, df)
    preds[name] = (pred, proba)
    results[name] = evaluate(df.label_binary, pred)

baseline_metrics = pd.DataFrame(results).T
baseline_metrics.insert(0, "modele", "TF-IDF + LogReg (entraine sans jailbreak)")
display(baseline_metrics.round(4))
baseline_metrics.to_csv("results/baseline_metrics.csv")

### Comment lire ce tableau
- **`test_seen`** : attaques de familles connues → on attend un bon score.
- **`test_unseen_family`** : la colonne **`detection_rate`** est la part des jailbreaks (jamais vus) détectés : c'est LE chiffre de généralisation.
  La colonne `fpr` mesure les faux positifs sur des prompts légitimes.

## 5. Ablation : l'effet de la famille retirée

On entraîne le **même** modèle sur un train qui **contient** les jailbreaks, puis on l'évalue sur la même `test_unseen_family`.
La différence montre exactement ce que coûte (ou non) le fait de ne pas avoir vu la famille.

*Les jailbreaks du train d'origine sont retrouvés dans `train_clean.csv`. Aucun texte de `test_unseen_family` ne fait partie de ce train : seules les lignes d'origine « train » sont utilisées.*

In [ ]:
train_full = pd.read_csv(f"{P}/train_clean.csv")
# Les jailbreaks du train d'origine ne doivent pas se retrouver dans le test_unseen : on les retire du test pour une comparaison honnête
orig_train_texts = set(train_full.text.astype(str).str.strip().str.lower())
test_unseen_fair = test_unseen[~test_unseen.text.astype(str).str.strip().str.lower().isin(orig_train_texts)]
print("test_unseen utilisable pour l'ablation :", len(test_unseen_fair), "lignes sur", len(test_unseen))

vec_all, clf_all = fit_baseline(train_full, **configs[best_name])
pred_all, _  = predict(vec_all, clf_all, test_unseen_fair)
pred_seen, _ = predict(vec, clf, test_unseen_fair)

abl = pd.DataFrame({
    "entrainement sans jailbreak (famille NON vue)": evaluate(test_unseen_fair.label_binary, pred_seen),
    "entrainement avec jailbreak (famille vue)":     evaluate(test_unseen_fair.label_binary, pred_all),
}).T
display(abl[["n", "detection_rate", "fpr", "f1", "TP", "FN", "FP", "TN"]].round(4))
abl.to_csv("results/baseline_ablation_family.csv")

## 6. Courbes et matrices de confusion

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name, df in zip(axes, ["validation (seen)", "test_seen", "test_unseen_family"],
                        [val_seen, test_seen, test_unseen]):
    pred, _ = preds[name]
    cm = confusion_matrix(df.label_binary, pred, labels=[0, 1])
    ConfusionMatrixDisplay(cm, display_labels=["légitime", "malveillant"]).plot(ax=ax, colorbar=False, values_format="d")
    ax.set_title(name)
plt.tight_layout()
plt.savefig("results/figures/baseline_confusion_matrices.png", dpi=150)
plt.show()

## 7. Analyse des erreurs

On regarde à la main ce que le modèle rate : c'est ce que le jury demandera.

In [ ]:
pd.set_option("display.max_colwidth", 200)

# (a) Attaques manquées sur la famille NON vue
pred_u, proba_u = preds["test_unseen_family"]
u = test_unseen.assign(pred=pred_u, proba=proba_u)
fn_unseen = u[(u.label_binary == 1) & (u.pred == 0)]
print("Jailbreaks manqués (famille non vue) :", len(fn_unseen), "sur", int((u.label_binary == 1).sum()))
display(fn_unseen[["text", "source", "proba"]].head(5))

# (b) Faux positifs sur les prompts légitimes
fp_unseen = u[(u.label_binary == 0) & (u.pred == 1)]
print("Faux positifs :", len(fp_unseen), "sur", int((u.label_binary == 0).sum()))
display(fp_unseen[["text", "source", "proba"]].head(5))

# (c) Attaques manquées sur le test_seen
pred_s, proba_s = preds["test_seen"]
s = test_seen.assign(pred=pred_s, proba=proba_s)
fn_seen = s[(s.label_binary == 1) & (s.pred == 0)]
print("Attaques manquées (test_seen) :", len(fn_seen))
print(fn_seen.family.value_counts().to_string())
display(fn_seen[["text", "family", "source", "proba"]].head(5))

fn_unseen.to_csv("results/errors_missed_unseen.csv", index=False)
fp_unseen.to_csv("results/errors_false_positives.csv", index=False)
fn_seen.to_csv("results/errors_missed_seen.csv", index=False)

### Mots les plus importants pour le modèle (interprétation)

In [ ]:
feat = np.array(vec.get_feature_names_out())
coef = clf.coef_[0]
top_mal = feat[np.argsort(coef)[-15:][::-1]]
top_ben = feat[np.argsort(coef)[:15]]
print("Indices de MALVEILLANCE :", list(top_mal))
print("Indices de LÉGITIMITÉ   :", list(top_ben))

## 8. Sauvegarde du modèle (pour réutilisation par les autres membres)

In [ ]:
joblib.dump(vec, "models/tfidf_vectorizer_seen.pkl")
joblib.dump(clf, "models/logistic_regression_seen.pkl")
print("✓ modèles sauvegardés dans models/")

## Conclusion
- La baseline est entraînée **sans** la famille `jailbreak` et évaluée sur 3 ensembles bien séparés.
- Les chiffres de référence (détection, faux positifs, F1) sont dans `results/baseline_metrics.csv`.
- L'ablation « avec / sans jailbreak dans l'entraînement » (`results/baseline_ablation_family.csv`) quantifie l'effet de la famille non vue.
- **Le modèle deep learning (Membre 2) devra faire mieux que ces chiffres sur `test_unseen_family`** pour justifier sa complexité.
- Voir le notebook 05 pour vérifier que ces scores ne viennent pas de raccourcis (longueur, langue, source).